# Pasar los CSV a SQL

Python solo lee los CSV y escribe sus filas tal cual en `../sql/insertar_datos.sql` (una tabla `csv_...` por archivo). El reparto en las tablas definitivas lo hace SQL, con JOINs, en `repartir_datos.sql`.

En pgAdmin (Query Tool sobre `bootcamp_oxbz`), en este orden:
1. `crear_tablas.sql`
2. `insertar_datos.sql`
3. `repartir_datos.sql`

In [ ]:
from pathlib import Path
import pandas as pd

# Cada CSV de ../data se lee tal cual: csv_clase_1, ..., csv_claustro (columnas en minúsculas, todo como texto)
tablas = {f"csv_{f.stem}": pd.read_csv(f, sep=";", dtype=str).rename(columns=str.lower)
          for f in sorted(Path("../data").glob("*.csv"))}

In [ ]:
def a_sql(valor):
    """Convierte un valor de Python en texto SQL."""
    return "'" + valor.replace("'", "''") + "'"   # todo es texto: entre comillas simples

lineas = ["BEGIN;"]
for nombre, df in tablas.items():
    lineas.append(f"DROP TABLE IF EXISTS {nombre};\nCREATE TABLE {nombre} ({', '.join(c + ' text' for c in df.columns)});")
    filas = ["  (" + ", ".join(a_sql(v) for v in fila) + ")"
             for fila in df.itertuples(index=False, name=None)]
    lineas.append(f"INSERT INTO {nombre} ({', '.join(df.columns)}) VALUES\n" + ",\n".join(filas) + ";")
lineas.append("COMMIT;")

with open("../sql/insertar_datos.sql", "w", encoding="utf-8") as f:
    f.write("\n\n".join(lineas))